In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from scipy.integrate import solve_ivp
from scipy.stats import pearsonr

# ── Reproducibility ───────────────────────────────────────────────────────────
rng = np.random.default_rng(seed=42)

# ── Global constants ──────────────────────────────────────────────────────────
N_SPECIES    = 40          # 1 pathogen + 9 specialists + 10 generalists
N_RESOURCES  = 10
N_GENES      = 6
N_BROTHS     = 10

PATHOGEN_IDX = 0
N_COMMUNITY  = N_SPECIES - 1
N_SPECIALIST = N_COMMUNITY // 2            # 9  → indices 1–9
N_GENERALIST = N_COMMUNITY - N_SPECIALIST  # 10 → indices 10–19

RESOURCE_NAMES = [f"c{k+1}" for k in range(N_RESOURCES)]
SPECIES_NAMES  = [f"Species_{i+1:02d}" for i in range(N_SPECIES)]
BROTH_NAMES    = [f"Broth_{b+1:02d}"   for b in range(N_BROTHS)]

OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads"

TYPE_COLORS = {
    "pathogen"   : "#E53935",
    "specialist" : "#1E88E5",
    "generalist" : "#43A047",
}


# =============================================================================
# 1.  SPECIES TRAITS
# =============================================================================

def build_species_traits():
    """Binary genomes → sigmoid uptake rates."""
    genes = rng.integers(0, 2, size=(N_SPECIES, N_GENES))
    W     = rng.normal(0.0, 0.5, size=(N_GENES, N_RESOURCES))

    c_matrix           = np.zeros((N_SPECIES, N_RESOURCES))
    species_type       = []
    preferred_nutrient = []

    for i in range(N_SPECIES):
        if i == PATHOGEN_IDX:
            bias = rng.uniform(-0.8, -0.3, size=N_RESOURCES)
            raw  = genes[i].astype(float) @ W + bias
            c_matrix[i] = 1.0 / (1.0 + np.exp(-raw))
            species_type.append("pathogen")
            preferred_nutrient.append(-1)

        elif 1 <= i <= N_SPECIALIST:
            pref = (i - 1) % N_RESOURCES
            bias = rng.uniform(-2.0, -1.0, size=N_RESOURCES)
            bias[pref] += rng.uniform(2.5, 4.0)
            raw  = genes[i].astype(float) @ W + bias
            c_matrix[i] = 1.0 / (1.0 + np.exp(-raw))
            species_type.append("specialist")
            preferred_nutrient.append(pref)

        else:
            bias = rng.uniform(-0.8, -0.3, size=N_RESOURCES)
            raw  = genes[i].astype(float) @ W + bias
            c_matrix[i] = 1.0 / (1.0 + np.exp(-raw))
            species_type.append("generalist")
            preferred_nutrient.append(-1)

    return genes, c_matrix, species_type, preferred_nutrient


# =============================================================================
# 2.  BROTH ENVIRONMENTS  ← REVISED
# =============================================================================

def build_broths():
    """
    Each broth now has 1–2 nutrients that are genuinely ABSENT (R0 ≈ 0).
    This means cross-feeding is the ONLY way those nutrients enter the system,
    making species-specific D_i matrices ecologically consequential.

    Design rules
    ────────────
    • Always 1 dominant nutrient   (high R0, 3–6× amplified)
    • Always 1–2 absent nutrients  (R0 = 0; drawn without replacement from
                                    the non-dominant slots)
    • Remaining nutrients present  (R0 drawn from U(0.5, 2.0))

    This guarantees at least one nutrient is missing in every broth so that
    cross-feeding products are not redundant.
    """
    broths = []
    for b in range(N_BROTHS):
        dominant = int(rng.integers(0, N_RESOURCES))

        # Pick 1 or 2 absent nutrients from the non-dominant slots
        non_dominant = [k for k in range(N_RESOURCES) if k != dominant]
        n_absent     = int(rng.integers(2, 5))          # 1 or 2 absent nutrients
        absent_set   = set(rng.choice(non_dominant, size=n_absent, replace=False).tolist())

        R0 = np.zeros(N_RESOURCES)
        for k in range(N_RESOURCES):
            if k == dominant:
                R0[k] = rng.uniform(0.5, 2.0) * rng.uniform(3.0, 6.0)
            elif k in absent_set:
                R0[k] = 0.0          # genuinely absent — only cross-feeding can supply it
            else:
                R0[k] = rng.uniform(0.5, 2.0)

        delta = float(rng.uniform(0.05, 0.20))
        w     = rng.uniform(0.6, 1.2, size=N_RESOURCES)
        l     = rng.uniform(0.1, 0.3, size=N_RESOURCES)

        broths.append({
            "name"       : BROTH_NAMES[b],
            "delta"      : delta,
            "R0"         : R0,
            "w"          : w,
            "l"          : l,
            "dominant"   : dominant,
            "absent"     : sorted(absent_set),      # NEW: track which nutrients are absent
            "members"    : list(range(N_SPECIES)),
        })
    return broths


# =============================================================================
# 3.  SPECIES-SPECIFIC CROSS-FEEDING MATRICES
# =============================================================================

def build_species_crossfeed_matrices(species_type):
    """One Dirichlet-drawn D_i per species."""
    D_list = []
    for i in range(N_SPECIES):
        D_i = np.zeros((N_RESOURCES, N_RESOURCES))
        for beta in range(N_RESOURCES):
            alpha_vec = np.ones(N_RESOURCES)
            if species_type[i] == "specialist":
                pref = (i - 1) % N_RESOURCES
                if beta == pref:
                    alpha_vec[pref] = 0.2
                    for k in range(N_RESOURCES):
                        if k != pref:
                            alpha_vec[k] = rng.uniform(2.0, 5.0)
                else:
                    alpha_vec[beta] = 0.5
                    for k in range(N_RESOURCES):
                        if k != beta:
                            alpha_vec[k] = rng.uniform(0.5, 2.0)
            elif species_type[i] == "pathogen":
                alpha_vec = rng.uniform(0.5, 3.0, size=N_RESOURCES)
            else:
                alpha_vec = rng.uniform(0.8, 1.5, size=N_RESOURCES)
            D_i[:, beta] = rng.dirichlet(alpha_vec)
        D_list.append(D_i)
    return D_list


# =============================================================================
# 4.  ODE
# =============================================================================

def micrm_ode(t, y, c, w, l, m, g, delta, R0, D_list_sub):
    n_sp  = c.shape[0]
    n_res = c.shape[1]
    N = np.maximum(y[:n_sp], 0.0)
    R = np.maximum(y[n_sp:], 0.0)

    energy_intake = c @ (w * (1.0 - l) * R)
    net_growth    = g * (energy_intake - m)
    dN = N * net_growth - delta * N

    dilution_net = delta * (R0 - R)
    uptake       = (c.T @ N) * R

    eps = 1e-12
    crossfeed = np.zeros(n_res)
    for i in range(n_sp):
        for beta in range(n_res):
            flux_i_beta = l[beta] * c[i, beta] * N[i] * R[beta]
            for alpha in range(n_res):
                crossfeed[alpha] += (D_list_sub[i][alpha, beta]
                                     * w[beta] / (w[alpha] + eps)
                                     * flux_i_beta)
    dR = dilution_net - uptake + crossfeed
    return np.concatenate([dN, dR])


# =============================================================================
# 5.  ODE RUNNER
# =============================================================================

def run_micrm(c_full, w, l, m_full, g_full, delta, R0, D_list,
              species_subset, t_end=500.0):
    c_sub      = c_full[species_subset, :]
    m_sub      = m_full[species_subset]
    g_sub      = g_full[species_subset]
    D_list_sub = [D_list[i] for i in species_subset]
    n_sub      = len(species_subset)

    y0 = np.concatenate([np.full(n_sub, 0.01), R0.copy()])
    sol = solve_ivp(
        fun    = micrm_ode,
        t_span = (0.0, t_end),
        y0     = y0,
        args   = (c_sub, w, l, m_sub, g_sub, delta, R0, D_list_sub),
        method = "RK45",
        t_eval = np.linspace(0, t_end, 600),
        rtol   = 1e-6,
        atol   = 1e-8,
    )
    y_final = sol.y[:, -1]
    N_ss    = np.maximum(y_final[:n_sub], 0.0)
    R_final = np.maximum(y_final[n_sub:], 0.0)

    N_out = np.zeros(N_SPECIES)
    for local_idx, global_idx in enumerate(species_subset):
        N_out[global_idx] = N_ss[local_idx]
    return N_out, R_final, sol


# =============================================================================
# 6.  METRIC FUNCTIONS
# =============================================================================

def calc_pairwise_suppression(N_pathogen_pair, N_pathogen_solo):
    eps = 1e-10
    if N_pathogen_solo < eps:
        return 0.0
    raw   = 1.0 - (N_pathogen_pair / (N_pathogen_solo + eps))
    
    return float(raw)


def calc_community_suppression(N_pathogen_comm, N_pathogen_solo):
    eps = 1e-10
    if N_pathogen_solo < eps:
        return 0.0
    raw   = 1.0 - (N_pathogen_comm / (N_pathogen_solo + eps))
    
    return float(raw)


# =============================================================================
# 7.  MAIN DATASET GENERATOR
# =============================================================================

def generate_dataset():
    print("\n[1] Generating species traits …")
    genes, c_matrix, species_type, preferred_nutrient = build_species_traits()
    m = rng.uniform(0.2, 0.6, size=N_SPECIES)
    g = rng.uniform(0.8, 1.2, size=N_SPECIES)

    print("    Species composition:")
    for i in range(N_SPECIES):
        pn = f"prefers c{preferred_nutrient[i]+1}" if preferred_nutrient[i] >= 0 else "generalist"
        print(f"      {SPECIES_NAMES[i]:15s}  [{species_type[i]:11s}]  {pn}")

    print("\n[2] Building species-specific D_i matrices …")
    D_list = build_species_crossfeed_matrices(species_type)

    print("\n[3] Generating broth environments …")
    broths = build_broths()
    for b in broths:
        absent_str = [RESOURCE_NAMES[k] for k in b["absent"]]
        print(f"    {b['name']}: dominant={RESOURCE_NAMES[b['dominant']]}, "
              f"absent={absent_str}, delta={b['delta']:.3f}, R0={np.round(b['R0'],2)}")

    # ── Pathogen solo ─────────────────────────────────────────────────────
    print("\n[4] Computing pathogen solo abundances …")
    pathogen_solo = {}
    for broth in broths:
        N_s, _, _ = run_micrm(
            c_full=c_matrix, w=broth["w"], l=broth["l"], m_full=m, g_full=g,
            delta=broth["delta"], R0=broth["R0"], D_list=D_list,
            species_subset=[PATHOGEN_IDX], t_end=400.0,
        )
        pathogen_solo[broth["name"]] = float(N_s[PATHOGEN_IDX])
        print(f"    {broth['name']}: N_pathogen_solo = {N_s[PATHOGEN_IDX]:.4f}")

    # ── Pairwise ODE runs ─────────────────────────────────────────────────
    print("\n[5] Running pairwise ODE (species_i + pathogen) …")
    pairwise_rows   = []
    pairwise_matrix = {}

    for broth in broths:
        bname       = broth["name"]
        N_path_solo = pathogen_solo[bname]
        pairwise_matrix[bname] = {}

        for sp_idx in range(1, N_SPECIES):
            N_pair, _, _ = run_micrm(
                c_full=c_matrix, w=broth["w"], l=broth["l"], m_full=m, g_full=g,
                delta=broth["delta"], R0=broth["R0"], D_list=D_list,
                species_subset=[sp_idx, PATHOGEN_IDX], t_end=400.0,
            )
            N_pathogen_pair = float(N_pair[PATHOGEN_IDX])
            pw_score = calc_pairwise_suppression(N_pathogen_pair, N_path_solo)
            pairwise_matrix[bname][sp_idx] = pw_score

            row = {
                "species_id"        : SPECIES_NAMES[sp_idx],
                "broth_id"          : bname,
                "species_type"      : species_type[sp_idx],
                "dominant_nutrient" : RESOURCE_NAMES[broth["dominant"]],
                "absent_nutrients"  : str([RESOURCE_NAMES[k] for k in broth["absent"]]),  # NEW
                "delta_per_h"       : round(broth["delta"], 4),
                "maintenance_cost"  : round(float(m[sp_idx]), 4),
            }
            for g_idx in range(N_GENES):
                row[f"gene_{g_idx+1}"] = int(genes[sp_idx, g_idx])
            for k in range(N_RESOURCES):
                row[f"uptake_{RESOURCE_NAMES[k]}"] = round(float(c_matrix[sp_idx, k]), 4)

            # NEW: uptake of the species for each absent nutrient
            # (a proxy for whether this species can exploit cross-fed products)
            for k in range(N_RESOURCES):
                row[f"R0_{RESOURCE_NAMES[k]}"] = round(float(broth["R0"][k]), 4)

            row["suppression_score"] = round(pw_score, 4)
            pairwise_rows.append(row)

        scores = [pairwise_matrix[bname][i] for i in range(1, N_SPECIES)]
        print(f"    {bname}: scores [{min(scores):.3f}, {max(scores):.3f}]  "
              f"path_solo={N_path_solo:.4f}")

    df = pd.DataFrame(pairwise_rows)

    # ── Full community ODE ────────────────────────────────────────────────
    print("\n[6] Running full community ODE (all 20 species) …")
    community_N   = {}
    comm_supp_map = {}

    for broth in broths:
        bname = broth["name"]
        N_comm, R_ss, sol_comm = run_micrm(
            c_full=c_matrix, w=broth["w"], l=broth["l"], m_full=m, g_full=g,
            delta=broth["delta"], R0=broth["R0"], D_list=D_list,
            species_subset=broth["members"], t_end=500.0,
        )
        community_N[bname]   = (N_comm, R_ss, sol_comm)
        comm_supp             = calc_community_suppression(
            N_comm[PATHOGEN_IDX], pathogen_solo[bname])
        comm_supp_map[bname] = comm_supp
        print(f"    {bname}: community suppression = {comm_supp:.3f}  "
              f"N_pathogen_comm = {N_comm[PATHOGEN_IDX]:.4f}")

    df["community_pathogen_suppression"] = df["broth_id"].map(comm_supp_map)

    # ── Final column order ────────────────────────────────────────────────
    gene_cols   = [f"gene_{g+1}"   for g in range(N_GENES)]
    uptake_cols = [f"uptake_{r}"   for r in RESOURCE_NAMES]
    R0_cols     = [f"R0_{r}"       for r in RESOURCE_NAMES]
    col_order = (
        ["species_id", "broth_id",
         "species_type", "dominant_nutrient", "absent_nutrients",
         "delta_per_h", "maintenance_cost"]
        + gene_cols + uptake_cols + R0_cols
        + ["suppression_score", "community_pathogen_suppression"]
    )
    df = df[col_order]

    print(f"\n  Dataset shape : {df.shape}")
    print(f"  suppression_score summary:\n{df['suppression_score'].describe().round(3)}")
    print(f"\n  Fraction suppressing pathogen (score > 0): "
          f"{(df['suppression_score'] > 0).mean():.2%}")

    return (df, genes, c_matrix, m, g, broths,
            D_list, species_type, preferred_nutrient,
            pathogen_solo, pairwise_matrix, community_N, comm_supp_map)


# =============================================================================
# 8.  FIGURES
# =============================================================================

def make_figures(df, genes, c_matrix, m, g, broths,
                 D_list, species_type, preferred_nutrient,
                 pathogen_solo, pairwise_matrix, community_N, comm_supp_map):

    print("\n[7] Generating figures …")

    # ── FIG 1: Full community ODE time series ─────────────────────────────
    b0     = broths[0]
    bname0 = b0["name"]
    _, _, sol = community_N[bname0]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for i in range(N_SPECIES):
        col = TYPE_COLORS[species_type[i]]
        lw  = 2.5 if i == PATHOGEN_IDX else 0.9
        zo  = 5   if i == PATHOGEN_IDX else 1
        axes[0].plot(sol.t, sol.y[i], color=col, lw=lw, alpha=0.75, zorder=zo)
    from matplotlib.lines import Line2D
    axes[0].legend(handles=[
        Line2D([0],[0], color=TYPE_COLORS["pathogen"],   lw=2.5, label="Pathogen"),
        Line2D([0],[0], color=TYPE_COLORS["specialist"], lw=1.5, label="Specialist"),
        Line2D([0],[0], color=TYPE_COLORS["generalist"], lw=1.5, label="Generalist"),
    ], fontsize=8)
    axes[0].set_xlabel("Time (h)")
    axes[0].set_ylabel("Abundance N_i")
    absent_str = [RESOURCE_NAMES[k] for k in b0["absent"]]
    axes[0].set_title(
        f"Community Dynamics — {bname0}\n"
        f"dominant={RESOURCE_NAMES[b0['dominant']]}, absent={absent_str}",
        fontweight="bold")
    axes[0].set_yscale("symlog", linthresh=0.001)
    axes[0].grid(True, alpha=0.3)

    res_colors = plt.cm.Set1(np.linspace(0, 1, N_RESOURCES))
    for k in range(N_RESOURCES):
        axes[1].plot(sol.t, sol.y[N_SPECIES + k],
                     color=res_colors[k], lw=2.5,
                     label=f"{RESOURCE_NAMES[k]}"
                           f"{' (absent)' if k in b0['absent'] else ''}")
    axes[1].set_xlabel("Time (h)")
    axes[1].set_ylabel("Nutrient concentration R_α")
    axes[1].set_title("Nutrient Dynamics\n(absent nutrients supplied only by cross-feeding)",
                      fontweight="bold")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)
    fig.suptitle("Figure 1: Full Community MiCRM Dynamics", fontsize=11, fontweight="bold")
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig1_ode_dynamics_v4.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig1_ode_dynamics_v3.png")

    # ── FIG 2: Genome + Uptake heatmaps ───────────────────────────────────
    row_colors = [TYPE_COLORS[t] for t in species_type]
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    fig.suptitle("Figure 2: Genome → Uptake Rate Mapping\n"
                 "(Row colour: red=Pathogen, blue=Specialist, green=Generalist)",
                 fontsize=12, fontweight="bold")
    sns.heatmap(genes.astype(float), ax=axes[0], cmap="Blues", vmin=0, vmax=1,
                xticklabels=[f"G{g_+1}" for g_ in range(N_GENES)],
                yticklabels=[f"Sp{i+1}" for i in range(N_SPECIES)],
                linewidths=0.3, linecolor="white",
                cbar_kws={"label": "Gene state (0/1)"})
    axes[0].set_title("Binary Genomes", fontweight="bold")
    for tick, color in zip(axes[0].get_yticklabels(), row_colors):
        tick.set_color(color)
    axes[0].tick_params(axis="y", labelsize=7)

    sns.heatmap(c_matrix, ax=axes[1], cmap="YlOrRd", vmin=0, vmax=1,
                xticklabels=RESOURCE_NAMES,
                yticklabels=[f"Sp{i+1}" for i in range(N_SPECIES)],
                linewidths=0.3, linecolor="white",
                cbar_kws={"label": "Uptake rate c_iα"})
    axes[1].set_title("Uptake Rates c_iα\n(Specialists show column dominance)",
                      fontweight="bold")
    for tick, color in zip(axes[1].get_yticklabels(), row_colors):
        tick.set_color(color)
    axes[1].tick_params(axis="y", labelsize=7)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig2_genome_uptake_v4.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig2_genome_uptake_v3.png")

    # ── FIG 3: Species-specific D_i matrices ──────────────────────────────
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle("Figure 3: Species-Specific Cross-Feeding Matrices D_i\n"
                 "(rows = output nutrient, cols = consumed nutrient)",
                 fontsize=12, fontweight="bold")
    for ax_idx, sp_idx in enumerate([0, 1, 2, 3, 4, 10, 11, 12]):
        sns.heatmap(D_list[sp_idx], ax=axes.flatten()[ax_idx],
                    cmap="YlOrRd", vmin=0, vmax=0.8,
                    xticklabels=[f"in {r}" for r in RESOURCE_NAMES],
                    yticklabels=[f"out {r}" for r in RESOURCE_NAMES],
                    annot=True, fmt=".2f", linewidths=0.5,
                    cbar=(ax_idx == 7))
        axes.flatten()[ax_idx].set_title(
            f"{SPECIES_NAMES[sp_idx]} [{species_type[sp_idx]}]",
            fontweight="bold", color=TYPE_COLORS[species_type[sp_idx]])
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig3_species_crossfeed_matrices_v4.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig3_species_crossfeed_matrices_v3.png")

    # ── FIG 4: Average D by type + feed concentrations ────────────────────
    D_avg = {}
    for t in ["pathogen", "specialist", "generalist"]:
        mats = [D_list[i] for i in range(N_SPECIES) if species_type[i] == t]
        D_avg[t] = np.mean(mats, axis=0)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle("Figure 4: Average Cross-Feeding D by Species Type & Feed Concentrations\n"
                 "(black zeros = absent nutrients in R0)",
                 fontsize=12, fontweight="bold")
    for ax, (t, D), col in zip(axes[:3], D_avg.items(),
                                 ["#E53935", "#1E88E5", "#43A047"]):
        sns.heatmap(D, ax=ax, cmap="Blues", vmin=0, vmax=0.8,
                    xticklabels=[f"in {r}" for r in RESOURCE_NAMES],
                    yticklabels=[f"out {r}" for r in RESOURCE_NAMES],
                    annot=True, fmt=".2f", linewidths=0.5, cbar=False)
        ax.set_title(f"{t.capitalize()} (avg D)", fontweight="bold", color=col)
    R0_arr = np.array([b["R0"] for b in broths])
    # Mask zeros so they stand out visually
    mask_zero = R0_arr == 0
    sns.heatmap(R0_arr, ax=axes[3], cmap="YlOrRd", mask=~mask_zero,
                xticklabels=RESOURCE_NAMES,
                yticklabels=[b["name"].replace("Broth_", "B") for b in broths],
                vmin=0, vmax=R0_arr.max(),
                annot=R0_arr, fmt=".1f", linewidths=0.5,
                cbar=False)
    sns.heatmap(R0_arr, ax=axes[3], cmap="YlOrRd", mask=mask_zero,
                xticklabels=RESOURCE_NAMES,
                yticklabels=[b["name"].replace("Broth_", "B") for b in broths],
                vmin=0, vmax=R0_arr.max(),
                annot=R0_arr, fmt=".1f", linewidths=0.5,
                cbar_kws={"label": "Feed concentration R⁰  (0 = absent)"},
                annot_kws={"color": "black"})
    axes[3].set_title("Feed Concentrations R⁰ per Broth\n(0.0 = nutrient absent)", fontweight="bold")
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig4_crossfeed_type_avg_supply_v4.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig4_crossfeed_type_avg_supply_v3.png")

    # ── FIG 5: Solo vs community abundances ───────────────────────────────
    print("    (running solo ODE for Fig 5 abundance comparison …)")
    solo_N = {}
    for sp_idx in range(1, N_SPECIES):
        vals = []
        for broth in broths:
            N_s, _, _ = run_micrm(
                c_full=c_matrix, w=broth["w"], l=broth["l"], m_full=m, g_full=g,
                delta=broth["delta"], R0=broth["R0"], D_list=D_list,
                species_subset=[sp_idx], t_end=400.0,
            )
            vals.append(float(N_s[sp_idx]))
        solo_N[SPECIES_NAMES[sp_idx]] = np.mean(vals)

    comm_N_mean = {}
    for sp_idx in range(1, N_SPECIES):
        vals = [float(community_N[broth["name"]][0][sp_idx]) for broth in broths]
        comm_N_mean[SPECIES_NAMES[sp_idx]] = np.mean(vals)

    sp_names_comm = SPECIES_NAMES[1:]
    solo_vals  = [solo_N[s]      for s in sp_names_comm]
    comm_vals  = [comm_N_mean[s] for s in sp_names_comm]
    bar_colors = [TYPE_COLORS[species_type[i]] for i in range(1, N_SPECIES)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Figure 5: Solo vs Full-Community Steady-State Abundances",
                 fontsize=12, fontweight="bold")
    x = np.arange(len(sp_names_comm))
    axes[0].bar(x - 0.2, solo_vals, width=0.38, color="#B0BEC5", alpha=0.9, label="Solo")
    axes[0].bar(x + 0.2, comm_vals, width=0.38, color=bar_colors, alpha=0.85)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(
        [s.replace("Species_", "S") for s in sp_names_comm], rotation=70, fontsize=7)
    axes[0].set_ylabel("Mean steady-state abundance")
    axes[0].set_title("Solo vs Full Community\n(community bar = species type colour)",
                      fontweight="bold")
    from matplotlib.patches import Patch
    axes[0].legend(handles=[
        Patch(color="#B0BEC5",                  label="Solo"),
        Patch(color=TYPE_COLORS["specialist"],  label="Specialist (comm)"),
        Patch(color=TYPE_COLORS["generalist"],  label="Generalist (comm)"),
    ], fontsize=7)
    axes[0].grid(True, alpha=0.3, axis="y")

    for stype, col in [("specialist", TYPE_COLORS["specialist"]),
                       ("generalist", TYPE_COLORS["generalist"])]:
        idx = [i for i, _ in enumerate(sp_names_comm)
               if species_type[i + 1] == stype]
        axes[1].scatter([solo_vals[i] for i in idx],
                        [comm_vals[i] for i in idx],
                        c=col, s=55, alpha=0.7, edgecolors="none",
                        label=stype.capitalize(), zorder=3)
    lim = max(max(solo_vals), max(comm_vals)) * 1.05
    axes[1].plot([0, lim], [0, lim], "k--", lw=1.5, label="No change")
    axes[1].set_xlabel("N_solo")
    axes[1].set_ylabel("N_community")
    axes[1].set_title("Solo vs Community (colour = type)\nAbove diagonal = community benefit",
                      fontweight="bold")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig5_solo_vs_community_v4.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig5_solo_vs_community_v3.png")

    # ── FIG 6: Community ecological summary ───────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle("Figure 6: Ecological Summary — Suppression & Traits",
                 fontsize=13, fontweight="bold")

    # 6a – community suppression per broth (now annotated with absent nutrients)
    broth_names = [b["name"] for b in broths]
    comm_supps  = [comm_supp_map[bn] for bn in broth_names]
    bar_obj = axes[0,0].bar(broth_names, comm_supps, color="#7B1FA2", alpha=0.8, edgecolor="white")
    for xi, (v, b) in enumerate(zip(comm_supps, broths)):
        absent_str = ",".join([RESOURCE_NAMES[k] for k in b["absent"]])
        axes[0,0].text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
        axes[0,0].text(xi, -0.06, f"−{absent_str}", ha="center", fontsize=6,
                       color="red", rotation=45)
    axes[0,0].set_ylabel("Community pathogen suppression")
    axes[0,0].set_title("Community Pathogen Suppression per Broth\n"
                         "(red labels = absent nutrients)", fontweight="bold")
    axes[0,0].tick_params(axis="x", rotation=45)
    axes[0,0].set_ylim(-0.12, 1.12)
    axes[0,0].grid(True, alpha=0.3, axis="y")

    # 6b – suppression_score distribution by species type
    for stype, col in TYPE_COLORS.items():
        if stype == "pathogen":
            continue
        sub = df[df["species_type"] == stype]["suppression_score"]
        axes[0,1].hist(sub, bins=20, color=col, alpha=0.55,
                       label=f"{stype.capitalize()} (n={len(sub)})", edgecolor="none")
    axes[0,1].axvline(0, color="black", lw=1.5, ls="--", label="Neutral")
    axes[0,1].set_xlabel("Pairwise suppression score")
    axes[0,1].set_ylabel("Count")
    axes[0,1].set_title("Pairwise Suppression Distribution\nby Species Type", fontweight="bold")
    axes[0,1].legend(fontsize=8)
    axes[0,1].grid(True, alpha=0.3)

    # 6c – maintenance cost vs mean pairwise suppression
    mean_pw  = df.groupby("species_id")["suppression_score"].mean()
    sp_order = SPECIES_NAMES[1:]
    m_vals   = [m[i] for i in range(1, N_SPECIES)]
    sc_col   = [TYPE_COLORS[species_type[i]] for i in range(1, N_SPECIES)]
    axes[1,0].scatter(m_vals, [mean_pw[s] for s in sp_order],
                      c=sc_col, s=70, alpha=0.8, edgecolors="k", lw=0.5)
    axes[1,0].set_xlabel("Maintenance cost m_i")
    axes[1,0].set_ylabel("Mean pairwise suppression (across broths)")
    axes[1,0].set_title("Maintenance Cost vs Mean Suppression", fontweight="bold")
    axes[1,0].grid(True, alpha=0.3)

    # 6d – D off-diagonal diversity vs mean suppression
    mean_off = []
    for i in range(1, N_SPECIES):
        D_i  = D_list[i]
        off  = [D_i[a, b] for a in range(N_RESOURCES)
                for b in range(N_RESOURCES) if a != b]
        mean_off.append(np.mean(off))
    axes[1,1].scatter(mean_off, [mean_pw[s] for s in sp_order],
                      c=sc_col, s=70, alpha=0.8, edgecolors="k", lw=0.5)
    axes[1,1].set_xlabel("Mean off-diagonal D_i flux\n(cross-feeding diversity)")
    axes[1,1].set_ylabel("Mean pairwise suppression")
    axes[1,1].set_title("D_i Cross-feeding Diversity vs Suppression\n"
                         "(expect stronger signal with absent nutrients)",
                         fontweight="bold")
    axes[1,1].grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig6_ecological_summary_v4.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig6_ecological_summary_v3.png")

    # ── FIG 7: Pairwise suppression heatmap + ranked bar ──────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    fig.suptitle("Figure 7: Pairwise Suppression of Pathogen\n"
                 "(Each cell = 2-species ODE, species_i + pathogen)",
                 fontsize=12, fontweight="bold")

    pivot_pw = df.pivot_table(index="species_id", columns="broth_id",
                              values="suppression_score", aggfunc="mean")
    pivot_pw = pivot_pw.loc[SPECIES_NAMES[1:]]
    pivot_pw.index   = [s.replace("Species_", "S") for s in pivot_pw.index]
    pivot_pw.columns = [c.replace("Broth_", "B")   for c in pivot_pw.columns]

    sns.heatmap(pivot_pw, ax=axes[0], cmap="RdYlGn", center=0,
                vmin=-0.3, vmax=1.0, linewidths=0.3, linecolor="lightgray",
                cbar_kws={"label": "Pairwise suppression score"},
                annot=True, fmt=".2f", annot_kws={"size": 6})
    axes[0].set_title("Pairwise Suppression Score\n"
                      "Green = suppresses pathogen | Red = facilitates",
                      fontweight="bold")
    for tick in axes[0].get_yticklabels():
        sp_short = tick.get_text()
        orig     = "Species_" + sp_short.replace("S", "")
        if orig in SPECIES_NAMES:
            tick.set_color(TYPE_COLORS[species_type[SPECIES_NAMES.index(orig)]])
    axes[0].tick_params(axis="y", labelsize=7)

    mean_pw_df = df.groupby(["species_id", "species_type"])[
        "suppression_score"].mean().reset_index()
    mean_pw_df = mean_pw_df.sort_values("suppression_score", ascending=True)
    axes[1].barh(range(len(mean_pw_df)), mean_pw_df["suppression_score"],
                 color=[TYPE_COLORS[t] for t in mean_pw_df["species_type"]],
                 alpha=0.85, edgecolor="white")
    axes[1].set_yticks(range(len(mean_pw_df)))
    axes[1].set_yticklabels(
        [s.replace("Species_", "S") for s in mean_pw_df["species_id"]], fontsize=8)
    axes[1].axvline(0, color="black", lw=1.5, ls="--")
    axes[1].set_xlabel("Mean pairwise suppression score (across all broths)")
    axes[1].set_title("Ranked Mean Pairwise Suppression", fontweight="bold")
    axes[1].grid(True, alpha=0.3, axis="x")
    from matplotlib.patches import Patch as P2
    axes[1].legend(handles=[
        P2(color=TYPE_COLORS["specialist"], label="Specialist"),
        P2(color=TYPE_COLORS["generalist"], label="Generalist"),
    ], fontsize=9)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig7_pairwise_suppression_heatmap_v4.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig7_pairwise_suppression_heatmap_v3.png")

    # ── FIG 8: Pairwise ODE time series for 6 representative species ───────
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    broth_pw = broths[0]
    absent_str = [RESOURCE_NAMES[k] for k in broth_pw["absent"]]
    fig.suptitle(f"Figure 8: Pairwise ODE Dynamics — Species_i + Pathogen ({broth_pw['name']})\n"
                 f"absent nutrients: {absent_str}",
                 fontsize=12, fontweight="bold")

    for ax_idx, sp_idx in enumerate([1, 3, 5, 10, 12, 15]):
        ax       = axes.flatten()[ax_idx]
        pair_sub = [sp_idx, PATHOGEN_IDX]
        c_sub    = c_matrix[pair_sub, :]
        D_sub    = [D_list[i] for i in pair_sub]

        sol_plot = solve_ivp(
            micrm_ode, (0, 400),
            np.concatenate([np.full(2, 0.01), broth_pw["R0"].copy()]),
            args=(c_sub, broth_pw["w"], broth_pw["l"],
                  m[pair_sub], g[pair_sub],
                  broth_pw["delta"], broth_pw["R0"], D_sub),
            t_eval=np.linspace(0, 400, 500),
            method="RK45", rtol=1e-6, atol=1e-8,
        )
        ax.plot(sol_plot.t, sol_plot.y[0],
                color=TYPE_COLORS[species_type[sp_idx]], lw=2,
                label=f"{SPECIES_NAMES[sp_idx]} [{species_type[sp_idx]}]")
        ax.plot(sol_plot.t, sol_plot.y[1],
                color=TYPE_COLORS["pathogen"], lw=2, ls="--", label="Pathogen")
        ax.set_yscale("symlog", linthresh=0.001)
        ax.set_xlabel("Time (h)", fontsize=8)
        ax.set_ylabel("Abundance", fontsize=8)
        ax.set_title(f"{SPECIES_NAMES[sp_idx]} [{species_type[sp_idx]}]",
                     fontweight="bold", color=TYPE_COLORS[species_type[sp_idx]])
        score_val = df.loc[
            (df["species_id"] == SPECIES_NAMES[sp_idx]) &
            (df["broth_id"] == broth_pw["name"]),
            "suppression_score"].values
        if len(score_val):
            ax.text(0.97, 0.97, f"suppression={score_val[0]:.3f}",
                    transform=ax.transAxes, ha="right", va="top", fontsize=8,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig8_pairwise_ode_dynamics_v4.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig8_pairwise_ode_dynamics_v3.png")

    # ── FIG 9: Pairwise vs community suppression ───────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Figure 9: Pairwise vs Community Suppression Comparison",
                 fontsize=12, fontweight="bold")

    for stype, col in TYPE_COLORS.items():
        if stype == "pathogen":
            continue
        sub = df[df["species_type"] == stype]
        axes[0].scatter(sub["suppression_score"],
                        sub["community_pathogen_suppression"],
                        c=col, s=40, alpha=0.55, edgecolors="none",
                        label=stype.capitalize())
    r, p = pearsonr(df["suppression_score"], df["community_pathogen_suppression"])
    axes[0].set_xlabel("Pairwise suppression score")
    axes[0].set_ylabel("Community pathogen suppression")
    axes[0].set_title(f"Pairwise vs Community\n(r = {r:.2f}, p = {p:.3f})",
                      fontweight="bold")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    broth_mean_pw   = df.groupby("broth_id")["suppression_score"].mean()
    broth_comm_supp = pd.Series(comm_supp_map)
    axes[1].scatter(broth_mean_pw.values,
                    broth_comm_supp[broth_mean_pw.index].values,
                    s=100, c="#7B1FA2", alpha=0.85, edgecolors="k", zorder=5)
    for bn in broth_mean_pw.index:
        axes[1].annotate(bn.replace("Broth_", "B"),
                         xy=(broth_mean_pw[bn], broth_comm_supp[bn]),
                         xytext=(3, 3), textcoords="offset points", fontsize=7)
    axes[1].set_xlabel("Mean pairwise suppression (all species in broth)")
    axes[1].set_ylabel("Community pathogen suppression")
    axes[1].set_title("Broth-level: Mean Pairwise vs Community", fontweight="bold")
    axes[1].grid(True, alpha=0.3)

    pivot_small = df.pivot_table(index="species_id", columns="broth_id",
                                  values="suppression_score", aggfunc="mean")
    pivot_small = pivot_small.loc[SPECIES_NAMES[1:]]
    pivot_small.index   = [s.replace("Species_", "S") for s in pivot_small.index]
    pivot_small.columns = [c.replace("Broth_", "B")   for c in pivot_small.columns]
    sns.heatmap(pivot_small, ax=axes[2], cmap="RdYlGn", center=0,
                vmin=-0.3, vmax=1.0, linewidths=0.2, linecolor="lightgray",
                cbar_kws={"label": "Pairwise suppression"})
    axes[2].set_title("Pairwise Suppression Heatmap\n(species × broth)", fontweight="bold")
    axes[2].tick_params(axis="y", labelsize=7)

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig9_pairwise_vs_community_v4.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig9_pairwise_vs_community_v3.png")

    print("\n  All figures saved.")


# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    (df, genes, c_matrix, m, g, broths,
     D_list, species_type, preferred_nutrient,
     pathogen_solo, pairwise_matrix,
     community_N, comm_supp_map) = generate_dataset()

    out_path = f"{OUTPUT_DIR}/micrm_pairwise_dataset_v4.csv"
    df.to_csv(out_path, index=False)
    print(f"\n[Done] Dataset saved → micrm_pairwise_dataset_v3.csv  shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")

    make_figures(df, genes, c_matrix, m, g, broths,
                 D_list, species_type, preferred_nutrient,
                 pathogen_solo, pairwise_matrix,
                 community_N, comm_supp_map)

    print("\n[Sample rows]:")
    print(df[[
        "species_id", "broth_id", "species_type",
        "dominant_nutrient", "absent_nutrients", "delta_per_h", "maintenance_cost",
        "gene_1", "gene_2", "uptake_c1", "uptake_c2",
        "suppression_score", "community_pathogen_suppression",
    ]].head(10).to_string(index=False))


[1] Generating species traits …
    Species composition:
      Species_01       [pathogen   ]  generalist
      Species_02       [specialist ]  prefers c1
      Species_03       [specialist ]  prefers c2
      Species_04       [specialist ]  prefers c3
      Species_05       [specialist ]  prefers c4
      Species_06       [specialist ]  prefers c5
      Species_07       [specialist ]  prefers c6
      Species_08       [specialist ]  prefers c7
      Species_09       [specialist ]  prefers c8
      Species_10       [specialist ]  prefers c9
      Species_11       [specialist ]  prefers c10
      Species_12       [specialist ]  prefers c1
      Species_13       [specialist ]  prefers c2
      Species_14       [specialist ]  prefers c3
      Species_15       [specialist ]  prefers c4
      Species_16       [specialist ]  prefers c5
      Species_17       [specialist ]  prefers c6
      Species_18       [specialist ]  prefers c7
      Species_19       [specialist ]  prefers c8
      Spec